# 🧮 Ejercicios Teóricos — Capítulo 7: Cambios de Base

---

Estos ejercicios desarrollan el **razonamiento algebraico riguroso** sobre cambios de base.
Para cada ejercicio hay una demostración o cálculo manual, seguido de verificación en Python.

| Ejercicio | Tema |
|:---:|:---|
| 1 | Matrices similares tienen los mismos autovalores |
| 2 | Cambio de base entre dos bases no estándar |
| 3 | Calcular $A_\mathcal{B}$ paso a paso |
| 4 | Determinar si dos matrices son similares |
| 5 | Base en que una matriz diagonal se vuelve identidad |

## 🟦 Ejercicio 1: Matrices Similares Tienen los Mismos Autovalores

**Demostración:** Prueba que si $B = P^{-1}AP$ entonces $A$ y $B$ tienen el mismo polinomio
característico, y por tanto los mismos autovalores.

**Solución:**

$$\det(B - \lambda I) = \det(P^{-1}AP - \lambda I)$$

$$= \det(P^{-1}AP - \lambda P^{-1}IP)$$

$$= \det(P^{-1}(A - \lambda I)P)$$

$$= \det(P^{-1}) \cdot \det(A - \lambda I) \cdot \det(P)$$

$$= \frac{1}{\det(P)} \cdot \det(A - \lambda I) \cdot \det(P) = \det(A - \lambda I)$$

✅ Por tanto, $B$ y $A$ tienen el mismo polinomio característico, y los mismos autovalores.

> ⚠️ Nota: usamos $\det(P^{-1}) = 1/\det(P)$ y la multiplicatividad del determinante.

In [1]:
import numpy as np

# Verificación computacional del Ejercicio 1
A = np.array([[3., 1.],
              [2., 4.]])

# Cinco matrices de cambio de base distintas
matrices_P = [
    np.array([[1.,2.],[0.,1.]]),
    np.array([[1.,0.],[1.,1.]]),
    np.array([[2.,1.],[1.,2.]]),
    np.array([[1.,3.],[2.,1.]]),
    np.eye(2),  # trivial: P=I
]

evals_A = np.sort(np.linalg.eigvals(A))
print(f"Autovalores de A: {np.round(evals_A, 6)}")
print()
print(f"{'Matriz P':<25} {'Autovalores de B':<30} {'Iguales?'}")
print("-"*70)

for P in matrices_P:
    if abs(np.linalg.det(P)) < 1e-10:
        continue
    B = np.linalg.inv(P) @ A @ P
    evals_B = np.sort(np.linalg.eigvals(B))
    ok = np.allclose(evals_A, evals_B)
    print(f"P={P.tolist()!s:<25} λ={np.round(evals_B,4)!s:<30} {'✓' if ok else '✗'}")

print("\n✅ Los autovalores de A son invariantes bajo similitud — siempre los mismos.")

Autovalores de A: [2. 5.]

Matriz P                  Autovalores de B               Iguales?
----------------------------------------------------------------------
P=[[1.0, 2.0], [0.0, 1.0]]  λ=[2. 5.]                        ✓
P=[[1.0, 0.0], [1.0, 1.0]]  λ=[2. 5.]                        ✓
P=[[2.0, 1.0], [1.0, 2.0]]  λ=[2. 5.]                        ✓
P=[[1.0, 3.0], [2.0, 1.0]]  λ=[2. 5.]                        ✓
P=[[1.0, 0.0], [0.0, 1.0]]  λ=[2. 5.]                        ✓

✅ Los autovalores de A son invariantes bajo similitud — siempre los mismos.


## 🟩 Ejercicio 2: Cambio de Base entre Dos Bases No Estándar

Dadas dos bases de $\mathbb{R}^2$:

$$\mathcal{B}_1 = \left\{\begin{pmatrix}1\\1\end{pmatrix}, \begin{pmatrix}1\\-1\end{pmatrix}\right\}, \quad
\mathcal{B}_2 = \left\{\begin{pmatrix}2\\1\end{pmatrix}, \begin{pmatrix}1\\3\end{pmatrix}\right\}$$

Encuentra la matriz $M$ que convierte coordenadas de $\mathcal{B}_1$ a $\mathcal{B}_2$.

**Solución teórica:**

1. $P_1$ = columnas de $\mathcal{B}_1$ → convierte de $\mathcal{B}_1$ a estándar.
2. $P_2$ = columnas de $\mathcal{B}_2$ → convierte de $\mathcal{B}_2$ a estándar.
3. Para ir de $\mathcal{B}_1$ a $\mathcal{B}_2$: primero a estándar ($P_1$), luego a $\mathcal{B}_2$ ($P_2^{-1}$).

$$M = P_2^{-1} P_1$$

Entonces: $[\mathbf{v}]_{\mathcal{B}_2} = P_2^{-1} P_1 \cdot [\mathbf{v}]_{\mathcal{B}_1}$.

In [2]:
import numpy as np

# Bases
P1 = np.column_stack([[1.,1.], [1.,-1.]])   # B1
P2 = np.column_stack([[2.,1.], [1.,3.]])    # B2

# Matriz de cambio de B1 a B2
M = np.linalg.inv(P2) @ P1
print("Cambio de B1 a B2 — M = P2^{-1} P1:")
print(np.round(M, 4))

# Verificar con un vector concreto
v = np.array([5., 2.])  # en coordenadas estándar

# Coordenadas en B1 y B2
v_B1 = np.linalg.inv(P1) @ v
v_B2 = np.linalg.inv(P2) @ v

print(f"\nVector estándar v = {v}")
print(f"[v]_B1 = {np.round(v_B1, 4)}")
print(f"[v]_B2 = {np.round(v_B2, 4)}")

# Verificar: M @ [v]_B1 debe dar [v]_B2
v_B2_check = M @ v_B1
print(f"\nM @ [v]_B1 = {np.round(v_B2_check, 4)}")
print(f"¿Igual a [v]_B2? {np.allclose(v_B2_check, v_B2)} ✓")

# Verificar también la inversa
M_inv = np.linalg.inv(P1) @ P2   # de B2 a B1
print(f"\nMatriz inversa (B2 → B1): M^{{-1}} = P1^{{-1}} P2")
print(np.round(M_inv, 4))
print(f"M @ M^{{-1}} = I: {np.allclose(M @ M_inv, np.eye(2))} ✓")

Cambio de B1 a B2 — M = P2^{-1} P1:
[[ 0.4  0.8]
 [ 0.2 -0.6]]

Vector estándar v = [5. 2.]
[v]_B1 = [3.5 1.5]
[v]_B2 = [ 2.6 -0.2]

M @ [v]_B1 = [ 2.6 -0.2]
¿Igual a [v]_B2? True ✓

Matriz inversa (B2 → B1): M^{-1} = P1^{-1} P2
[[ 1.5  2. ]
 [ 0.5 -1. ]]
M @ M^{-1} = I: True ✓


## 🟦 Ejercicio 3: Calcular $A_\mathcal{B}$ Paso a Paso

Sea $T: \mathbb{R}^2 \to \mathbb{R}^2$ con $A = \begin{pmatrix}0 & -1\\ 1 & 0\end{pmatrix}$ (rotación $90°$)
y la base $\mathcal{B} = \{\mathbf{b}_1, \mathbf{b}_2\}$ con $\mathbf{b}_1 = (1,2)^\top$, $\mathbf{b}_2 = (0,1)^\top$.

**Calcula $A_\mathcal{B}$ paso a paso:**

**Paso 1:** $T(\mathbf{b}_1) = A\begin{pmatrix}1\\2\end{pmatrix} = \begin{pmatrix}-2\\1\end{pmatrix}$

**Paso 2:** $T(\mathbf{b}_2) = A\begin{pmatrix}0\\1\end{pmatrix} = \begin{pmatrix}-1\\0\end{pmatrix}$

**Paso 3:** Expresar $T(\mathbf{b}_1)$ y $T(\mathbf{b}_2)$ en coordenadas $\mathcal{B}$:

Las columnas de $A_\mathcal{B}$ son $P^{-1}T(\mathbf{b}_i)$ para cada vector de base.

$$A_\mathcal{B} = P^{-1} \begin{bmatrix} T(\mathbf{b}_1) & T(\mathbf{b}_2)\end{bmatrix} = P^{-1}AP$$

In [3]:
import numpy as np

# Rotación 90°
A = np.array([[0., -1.],
              [1.,  0.]])

b1 = np.array([1., 2.])
b2 = np.array([0., 1.])
P = np.column_stack([b1, b2])
P_inv = np.linalg.inv(P)

print("Pasos del cálculo de A_B:")
print("\nPaso 1: Calcular T(b1) y T(b2)")
Tb1 = A @ b1
Tb2 = A @ b2
print(f"  T(b1) = A @ {b1} = {Tb1}")
print(f"  T(b2) = A @ {b2} = {Tb2}")

print("\nPaso 2: Expresar T(b1) y T(b2) en coordenadas B")
Tb1_B = P_inv @ Tb1
Tb2_B = P_inv @ Tb2
print(f"  [T(b1)]_B = P^{{-1}} T(b1) = {np.round(Tb1_B, 4)}")
print(f"  [T(b2)]_B = P^{{-1}} T(b2) = {np.round(Tb2_B, 4)}")

print("\nPaso 3: A_B tiene estas columnas")
A_B_manual = np.column_stack([Tb1_B, Tb2_B])
print(A_B_manual)

print("\nVerificación con fórmula directa P^{-1}AP:")
A_B_formula = P_inv @ A @ P
print(np.round(A_B_formula, 6))
print(f"\n¿Iguales? {np.allclose(A_B_manual, A_B_formula)} ✓")
print("\nLa fórmula A_B = P^{-1}AP es equivalente a calcular columna a columna.")

Pasos del cálculo de A_B:

Paso 1: Calcular T(b1) y T(b2)
  T(b1) = A @ [1. 2.] = [-2.  1.]
  T(b2) = A @ [0. 1.] = [-1.  0.]

Paso 2: Expresar T(b1) y T(b2) en coordenadas B
  [T(b1)]_B = P^{-1} T(b1) = [-2.  5.]
  [T(b2)]_B = P^{-1} T(b2) = [-1.  2.]

Paso 3: A_B tiene estas columnas
[[-2. -1.]
 [ 5.  2.]]

Verificación con fórmula directa P^{-1}AP:
[[-2. -1.]
 [ 5.  2.]]

¿Iguales? True ✓

La fórmula A_B = P^{-1}AP es equivalente a calcular columna a columna.


## 🟥 Ejercicio 4: ¿Son Similares Estas Matrices?

Determina en cada caso si las matrices son similares:

**Caso A:**
$$M_1 = \begin{pmatrix}2&0\\0&3\end{pmatrix}, \quad M_2 = \begin{pmatrix}3&0\\0&2\end{pmatrix}$$

**Caso B:**
$$M_3 = \begin{pmatrix}1&0\\0&1\end{pmatrix} = I, \quad M_4 = \begin{pmatrix}1&1\\0&1\end{pmatrix}$$

**Criterios útiles:**
- Matrices similares tienen los mismos autovalores (misma traza, mismo determinante).
- Si $A$ es diagonalizable y tiene autovalores distintos, es similar a la diagonal de sus autovalores.
- $I$ solo es similar a $I$ (prueba: $P^{-1}IP = I$).

In [4]:
import numpy as np

print("=" * 55)
print("CASO A: M1 = diag(2,3) y M2 = diag(3,2)")
M1 = np.diag([2., 3.])
M2 = np.diag([3., 2.])

print(f"  tr(M1) = {np.trace(M1)}, tr(M2) = {np.trace(M2)}")
print(f"  det(M1) = {np.linalg.det(M1)}, det(M2) = {np.linalg.det(M2)}")
lM1 = np.sort(np.linalg.eigvals(M1))
lM2 = np.sort(np.linalg.eigvals(M2))
print(f"  λ(M1) = {lM1}, λ(M2) = {lM2}")
print(f"  ¿Mismos autovalores? {np.allclose(lM1, lM2)} → SÍ son similares")

# Encontrar P explícitamente: M2 = P^{-1}M1P
P_caso_A = np.array([[0.,1.],[1.,0.]])  # permutación
check = np.linalg.inv(P_caso_A) @ M1 @ P_caso_A
print(f"  Con P = matriz permutación: P^{{-1}}M1P = {check.tolist()}  == M2: {np.allclose(check, M2)} ✓")

print()
print("=" * 55)
print("CASO B: I y M4 = [[1,1],[0,1]]")
M3 = np.eye(2)
M4 = np.array([[1.,1.],[0.,1.]])

print(f"  tr(I) = {np.trace(M3)}, tr(M4) = {np.trace(M4)}")
print(f"  det(I) = {np.linalg.det(M3)}, det(M4) = {np.linalg.det(M4)}")
lM3 = np.sort(np.linalg.eigvals(M3))
lM4 = np.sort(np.linalg.eigvals(M4))
print(f"  λ(I) = {lM3}, λ(M4) = {lM4}")

print("  Aunque tienen los mismos autovalores y traza/det iguales:")
print("  P^{-1}IP = P^{-1}P = I ≠ M4 para cualquier P")
print("  I solo es similar a I misma → NO son similares.")
print("  Esto muestra que mismos autovalores NO garantiza similitud (caso Jordan).")

CASO A: M1 = diag(2,3) y M2 = diag(3,2)
  tr(M1) = 5.0, tr(M2) = 5.0
  det(M1) = 6.0, det(M2) = 6.0
  λ(M1) = [2. 3.], λ(M2) = [2. 3.]
  ¿Mismos autovalores? True → SÍ son similares
  Con P = matriz permutación: P^{-1}M1P = [[3.0, 0.0], [0.0, 2.0]]  == M2: True ✓

CASO B: I y M4 = [[1,1],[0,1]]
  tr(I) = 2.0, tr(M4) = 2.0
  det(I) = 1.0, det(M4) = 1.0
  λ(I) = [1. 1.], λ(M4) = [1. 1.]
  Aunque tienen los mismos autovalores y traza/det iguales:
  P^{-1}IP = P^{-1}P = I ≠ M4 para cualquier P
  I solo es similar a I misma → NO son similares.
  Esto muestra que mismos autovalores NO garantiza similitud (caso Jordan).


## 🟨 Ejercicio 5: Base que Convierte una Diagonal en la Identidad

Sea $D = \begin{pmatrix}\lambda_1 & 0\\ 0 & \lambda_2\end{pmatrix}$ con $\lambda_1, \lambda_2 \neq 0$.

**¿Existe una base $\mathcal{B}$ tal que la representación de $D$ en $\mathcal{B}$ sea la identidad?**

**Solución:** Buscamos $P$ tal que $P^{-1}DP = I$, es decir, $D = PIP^{-1} = PP^{-1} = ...$
¡Eso no funciona! $P^{-1}DP = I \Rightarrow D = PIP^{-1} = I$ — solo si $D = I$.

Pero sí podemos encontrar $P$ tal que $P^{-1}DP$ sea cualquier diagonal similar.

**Alternativa:** Usar $\mathcal{B} = \{\lambda_1\mathbf{e}_1, \lambda_2\mathbf{e}_2\}$, es decir,
$P = \text{diag}(\lambda_1, \lambda_2) = D$ misma.

Entonces: $P^{-1}DP = D^{-1}DD = D^{-1}D = I$... ¡sí funciona!

$$D^{-1} \cdot D \cdot D = D \neq I \quad (\text{corrección: } P^{-1}DP = D^{-1}DD = D)$$

Corrección: $P = D$ implica $P^{-1}DP = D^{-1}DD = D$, no $I$.

**La base correcta:** Escalar los vectores estándar por $1/\lambda_i$:
$P = \text{diag}(1/\lambda_1, 1/\lambda_2)$, entonces $P^{-1} = D$ y $P^{-1}DP = D \cdot D \cdot D^{-1} \cdot D$...

> 🔑 **Resultado clave:** Solo $D = I$ es similar a $I$. Pero escalando la base por $\sqrt{\lambda_i}$
> podemos simplificar enormemente la representación.

In [5]:
import numpy as np

# Ejercicio 5: explorar qué bases simplifican una diagonal
lambda1, lambda2 = 4., 9.
D = np.diag([lambda1, lambda2])
print("Matriz diagonal D:")
print(D)

# Intento 1: P = D
P1 = D.copy()
D_B1 = np.linalg.inv(P1) @ D @ P1
print(f"\nP = D → P^{{-1}}DP = {np.round(D_B1, 4)} (sigue siendo D, no I)")

# Intento 2: P = raíz de D
P2 = np.diag([np.sqrt(lambda1), np.sqrt(lambda2)])
D_B2 = np.linalg.inv(P2) @ D @ P2
print(f"P = sqrt(D) → P^{{-1}}DP = {np.round(D_B2, 4)} (sigue siendo D — diagonales similares son solo las que tienen los mismos valores)")

# Verificar que D solo es similar a matrices con autovalores {4, 9}
print("\nLa clase de similitud de D son todas las matrices con λ₁=4, λ₂=9")
print("Por ejemplo:")
P3 = np.array([[1.,2.],[0.,1.]])
M = P3 @ D @ np.linalg.inv(P3)   # M = P3 D P3^{-1}
print(f"Con P3 = [[1,2],[0,1]]: M = P3 D P3^{{-1}} =\n{np.round(M, 4)}")
print(f"Autovalores de M: {np.sort(np.real(np.linalg.eigvals(M)))}")
print(f"¿Son iguales a {{4,9}}? {np.allclose(np.sort(np.real(np.linalg.eigvals(M))), [4.,9.])} ✓")

print("\nConclusión: la única diagonal similar a I = diag(1,1) es I misma.")
print("No existe P tal que P^{-1}DP = I para D ≠ I.")

Matriz diagonal D:
[[4. 0.]
 [0. 9.]]

P = D → P^{-1}DP = [[4. 0.]
 [0. 9.]] (sigue siendo D, no I)
P = sqrt(D) → P^{-1}DP = [[4. 0.]
 [0. 9.]] (sigue siendo D — diagonales similares son solo las que tienen los mismos valores)

La clase de similitud de D son todas las matrices con λ₁=4, λ₂=9
Por ejemplo:
Con P3 = [[1,2],[0,1]]: M = P3 D P3^{-1} =
[[ 4. 10.]
 [ 0.  9.]]
Autovalores de M: [4. 9.]
¿Son iguales a {4,9}? True ✓

Conclusión: la única diagonal similar a I = diag(1,1) es I misma.
No existe P tal que P^{-1}DP = I para D ≠ I.
